# Train the V1 representation model

Production training workflow for the variable-length representation learning model. The model learns normal context consistency with EMA latent prediction and normal population geometry with file-level contrastive alignment.

Key features:

- Direct parameters: configure training directly in the notebook via `TrainingParams` class arguments (e.g. `params = TrainingParams(batch_size=128, epochs=40, in_memory=True)`).
- Manual paths: run from the repository root and set `SRC_DIR`, `V1_DATA_ROOT`, and `V1_CHECKPOINT_PATH` in the first cells; the exact configured paths are used with fail-fast errors naming the variable to change.
- In-memory caching & streaming: set `in_memory=True` (recommended for $\le 25\text{K}$ files) to preload samples into Host RAM with full per-epoch shuffling, or `in_memory=False` for $\mathcal{O}(1)$ disk streaming on $100\text{K}+$ datasets.
- Progress tracking: integrated `tqdm` progress bars tracking stationary joint loss, raw/weighted components, latent norms, contrastive similarity, and validation metrics.
- Hardware: automatically accelerates training with CUDA when available (`torch.cuda.is_available()`), falling back to CPU.
- Training dynamics: trains with AdamW optimizer, cosine annealing learning rate scheduler, gradient clipping, EMA target updates, and progressive contrastive ramp-up with calibrated $\tau=0.2$.
- Stationary model selection & coherent checkpointing: evaluates models using stationary joint loss ($\lambda_{\max}$ target weighting) to prevent pre-warmup selection traps, synchronizing model weights, optimizer, scheduler, step counter, and normal reference bank atomically to `checkpoints/v1_representation.pt`.
- Resumption & safety: explicit `resume=True` support to prevent accidental overwrites or Frankenstein checkpoint states.


In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from itertools import islice
import json
import os
from pathlib import Path
import sys
import torch
from tqdm.auto import tqdm

# ---- Server paths: repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V1_REPO_ROOT to the checkout path.
V1_REPO_ROOT = os.environ.get('V1_REPO_ROOT', '.')
SRC_DIR = os.environ.get('V1_SRC_DIR', str(Path(V1_REPO_ROOT) / 'src'))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / 'representation').is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'representation' package. "
        f"Run from the repository root or set V1_REPO_ROOT / SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded representation modules from: {_source_dir}")

from representation import V1Config
from representation.checkpoint import load_checkpoint, save_checkpoint
from representation.data import FileDataset, collate_variable_files
from representation.inference import NormalReferenceBank, RepresentationInference
from representation.model import V1RepresentationModel
from representation.trainer import RepresentationTrainer
from synth.config import PatchConfig
from synth.patchify import Patchifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Hardware] Compute device:', device)
if device.type == 'cuda':
    print('[Hardware] CUDA device name:', torch.cuda.get_device_name(0))
    print('[Hardware] Allocated memory:', f"{torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")


In [ ]:
# ---- Server paths: dataset and checkpoint output (edit these one-line values; used exactly) ----
V1_DATA_ROOT = os.environ.get('V1_DATA_ROOT', 'data/generated/production')
V1_CHECKPOINT_PATH = os.environ.get('V1_CHECKPOINT_PATH', 'checkpoints/v1_representation_20260904_01.pt')

_default_data, _default_ckpt = V1_DATA_ROOT, V1_CHECKPOINT_PATH

@dataclass
class TrainingParams:
    """Unified training configuration (works from a repository checkout).
    
    Modify parameters directly here or pass keyword arguments to TrainingParams(...).
    """
    # Dataset and checkpoint paths (auto-resolved based on environment)
    data_root: str = _default_data
    checkpoint_path: str = _default_ckpt
    max_samples: int | None = int(os.environ['V1_MAX_SAMPLES']) if 'V1_MAX_SAMPLES' in os.environ else None
    resume: bool = os.environ.get('V1_RESUME', 'false').lower() in ('true', '1', 'yes')
    
    # In-memory RAM caching (recommended True for <= 25K files, False for 100K+ streaming)
    in_memory: bool = os.environ.get('V1_IN_MEMORY', 'true').lower() in ('true', '1', 'yes')
    
    # Training hyperparameters (optimized production defaults: B=128, 40 epochs)
    batch_size: int = int(os.environ.get('V1_BATCH_SIZE', '128'))
    epochs: int = int(os.environ.get('V1_EPOCHS', '40'))
    lr: float = float(os.environ.get('V1_LR', '1.5e-3'))
    weight_decay: float = float(os.environ.get('V1_WEIGHT_DECAY', '1e-4'))
    eta_min: float = float(os.environ.get('V1_ETA_MIN', '1e-5'))
    max_grad_norm: float = float(os.environ.get('V1_MAX_GRAD_NORM', '1.0'))
    
    # Contrastive schedule & loss calibration
    lambda_max: float = float(os.environ.get('V1_LAMBDA_MAX', '0.1'))  # validated user-editable parameter (conservative rerun default: 0.1)
    contrastive_warmup_steps: int = int(os.environ.get('V1_WARMUP_STEPS', str(196 * 5)))  # 5 epochs warmup
    contrastive_ramp_steps: int = int(os.environ.get('V1_RAMP_STEPS', str(196 * 5)))      # 5 epochs ramp
    contrastive_temperature: float = float(os.environ.get('V1_TEMPERATURE', '0.2'))       # calibrated tau=0.2
    
    # Model architecture
    d_model: int = int(os.environ.get('V1_D_MODEL', '128'))
    sequence_layers: int = int(os.environ.get('V1_SEQ_LAYERS', '4'))
    attention_heads: int = int(os.environ.get('V1_ATTN_HEADS', '4'))
    dropout: float = float(os.environ.get('V1_DROPOUT', '0.1'))
    
    # Selection policy
    selection_metric: str = os.environ.get('V1_SELECTION_METRIC', 'val_stationary_joint_loss')
    
    # Normal reference bank fitting
    max_ref_samples: int = int(os.environ.get('V1_MAX_REF_SAMPLES', '8192'))

    def __post_init__(self) -> None:
        if self.lambda_max < 0.0:
            raise ValueError("lambda_max must be non-negative")

# Direct instantiation
params = TrainingParams()

configured_root = Path(params.data_root).expanduser()
DATA_ROOT = configured_root if configured_root.is_absolute() else Path.cwd() / configured_root
MANIFEST_PATH = DATA_ROOT / 'manifest.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {MANIFEST_PATH}. Set V1_DATA_ROOT to the materialized dataset root (local alternative: data/generated/production).")
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_splits = ('train', 'val', 'test')
splits = manifest.get('splits')
if not isinstance(splits, dict):
    raise RuntimeError(f"V1 dataset manifest at {MANIFEST_PATH} has no split mapping; regenerate with uv run python -m synth.cli.")
missing_splits = [name for name in required_splits if name not in splits]
if missing_splits:
    raise RuntimeError(f"V1 dataset at {DATA_ROOT} is missing required splits: {', '.join(missing_splits)}. Regenerate with uv run python -m synth.cli.")

def load_split(name: str, limit: int = 4):
    entry = splits[name]
    if not isinstance(entry, dict) or entry.get('status') != 'complete':
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is not complete; rerun uv run python -m synth.cli --output {DATA_ROOT} --resume.")
    samples = list(islice(FileDataset(DATA_ROOT, split=name), limit))
    if not samples:
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is empty.")
    return samples

train_samples = load_split('train')
val_samples = load_split('val')
print('dataset root', DATA_ROOT, 'manifest counts', manifest['counts'])
print('train IDs', [sample.file_id for sample in train_samples], 'val IDs', [sample.file_id for sample in val_samples])
print(f"Configured parameters: epochs={params.epochs}, batch_size={params.batch_size}, lr={params.lr}, in_memory={params.in_memory}, resume={params.resume}, lambda_max={params.lambda_max}, selection_metric={params.selection_metric}")


In [ ]:
cfg = V1Config(
    n_channels=train_samples[0].C,
    patch_size=32,
    stride=16,
    d_model=params.d_model,
    sequence_layers=params.sequence_layers,
    attention_heads=params.attention_heads,
    dropout=params.dropout,
    contrastive_weight_max=params.lambda_max,
    contrastive_warmup_steps=params.contrastive_warmup_steps,
    contrastive_ramp_steps=params.contrastive_ramp_steps,
    contrastive_temperature=params.contrastive_temperature,
)
patchifier = Patchifier(PatchConfig(patch_size=32, stride=16, pad_end=True))

class StreamingBatchDataset:
    """Yield collated minibatches, with optional Host RAM caching and full shuffling."""
    def __init__(self, data_root, split, patchifier, config, b_size=32, max_count=None, base_seed=0, in_memory=True):
        self.data_root = data_root
        self.split = split
        self.patchifier = patchifier
        self.config = config
        self.batch_size = b_size
        self.max_count = max_count
        self.base_seed = base_seed
        self.in_memory = in_memory
        self.samples = None
        
        if in_memory:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            total = manifest['counts'][split] if self.max_count is None else min(self.max_count, manifest['counts'][split])
            self.samples = list(tqdm(iterator, total=total, desc=f"Loading {split} to RAM"))

    def __iter__(self):
        if self.samples is not None:
            indices = torch.randperm(len(self.samples)).tolist()
            chunk = []
            batch_idx = 0
            for idx in indices:
                chunk.append(self.samples[idx])
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )
        else:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            chunk = []
            batch_idx = 0
            for sample in iterator:
                chunk.append(sample)
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )

train_batches = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=3, in_memory=params.in_memory)
val_batches = StreamingBatchDataset(DATA_ROOT, 'val', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=4, in_memory=params.in_memory)

train_batch = next(iter(train_batches))
val_batch = next(iter(val_batches))
train_total = manifest['counts']['train'] if params.max_samples is None else min(params.max_samples, manifest['counts']['train'])
val_total = manifest['counts']['val'] if params.max_samples is None else min(params.max_samples, manifest['counts']['val'])
print(f"Datasets ready: train={train_total} files, val={val_total} files, batch_size={params.batch_size}, in_memory={params.in_memory}")
print('train signals', tuple(train_batch['signals'].shape), 'validation signals', tuple(val_batch['signals'].shape), 'mask composition', train_batch['mask_composition'])


In [ ]:
model = V1RepresentationModel(cfg, patchifier=patchifier)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=params.lr, weight_decay=params.weight_decay)

num_batches_per_epoch = (train_total + params.batch_size - 1) // params.batch_size
num_val_batches = (val_total + params.batch_size - 1) // params.batch_size
total_steps = max(1, params.epochs * num_batches_per_epoch)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=params.eta_min)

configured_ckpt = Path(params.checkpoint_path).expanduser()
checkpoint_path = configured_ckpt if configured_ckpt.is_absolute() else Path.cwd() / configured_ckpt
start_step = 0
if params.resume:
    if checkpoint_path.is_file():
        meta = load_checkpoint(checkpoint_path, model, optimizer=optimizer, scheduler=scheduler)
        start_step = int(meta["step"])
        print(f"[Resume] Successfully restored checkpoint from step {start_step} at {checkpoint_path} (lambda_max={params.lambda_max})")
    else:
        print(f"[Resume] Warning: Checkpoint {checkpoint_path} not found; starting fresh training.")
else:
    if checkpoint_path.is_file():
        print(f"[Training] Existing checkpoint at {checkpoint_path} will be superseded upon completion (resume=False, lambda_max={params.lambda_max}).")
    else:
        print(f"[Training] Starting fresh training (resume=False, lambda_max={params.lambda_max}).")

start_epoch = start_step // num_batches_per_epoch

trainer = RepresentationTrainer(
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    seed=cfg.seed,
    step=start_step,
    selection_metric=params.selection_metric,
    max_grad_norm=params.max_grad_norm,
)

if start_epoch >= params.epochs:
    print(f"[Resume] Configured total epochs ({params.epochs}) already completed at step {start_step} (epoch {start_epoch}); skipping training loop.")
else:
    print(f"Starting training: {params.epochs} total epochs (running epochs {start_epoch + 1}..{params.epochs}), {num_batches_per_epoch} batches/epoch ({total_steps} total steps, start_step={trainer.step}), lambda_max={params.lambda_max}, selection={trainer.selection_metric}, device={device}...")

history = []

if start_epoch < params.epochs:
    epoch_pbar = tqdm(range(start_epoch + 1, params.epochs + 1), desc="Training epochs")
    for epoch in epoch_pbar:
        batch_pbar = tqdm(train_batches, total=num_batches_per_epoch, desc=f"Epoch {epoch:2d}/{params.epochs}", leave=False)
        train_metrics = trainer.train_epoch(batch_pbar)
        metrics = dict(train_metrics)
        
        val_pbar = tqdm(val_batches, total=num_val_batches, desc="Validating", leave=False)
        val_metrics = trainer.validate(val_pbar)
        metrics.update({f"val_{k}": v for k, v in val_metrics.items()})
        
        trainer.record_eval(metrics, epoch=epoch)
        
        trainer.history[-1] = metrics
        history.append(metrics)
        
        epoch_pbar.set_postfix({
            "stat_joint": f"{metrics['stationary_joint_loss']:.4f}",
            "val_stat": f"{metrics['val_stationary_joint_loss']:.4f}",
            "pred": f"{metrics['prediction_loss']:.4f}",
            "val_pred": f"{metrics['val_prediction_loss']:.4f}",
            "cont": f"{metrics['contrastive_loss']:.4f}",
            "sim": f"{metrics.get('contrastive_sim', 0.0):.3f}",
            "margin": f"{metrics.get('contrastive_margin', 0.0):.3f}",
            "lambda": f"{metrics['effective_lambda']:.3f}/{params.lambda_max:.2f}",
        })

    if trainer.best_state is not None:
        restored = trainer.restore_best_state()
        print(f"[Selection] Restored coherent best state from Epoch {trainer.best_epoch} (Step {trainer.step}) with score {trainer.best_loss:.4f} ({trainer.selection_metric}, lambda_max={params.lambda_max}).")
    else:
        print(f"[Selection] Retaining active model state at Step {trainer.step} (selection={trainer.selection_metric}, lambda_max={params.lambda_max}).")
else:
    print(f"[Selection] Retaining restored checkpoint model state at Step {trainer.step} (selection={trainer.selection_metric}, lambda_max={params.lambda_max}).")

print('history', history)

# Fit normal reference bank on active restored model
model.eval()
max_ref = min(params.max_ref_samples, train_total)
ref_loader = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=max_ref, base_seed=99, in_memory=params.in_memory)
ref_total_batches = (max_ref + params.batch_size - 1) // params.batch_size
ref_embeddings_list = []
ref_pbar = tqdm(ref_loader, total=ref_total_batches, desc=f"Fitting reference bank ({max_ref} normal files)", leave=False)
with torch.no_grad():
    for batch in ref_pbar:
        dev_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        ref_out = model(dev_batch)
        ref_embeddings_list.append(ref_out['file_embedding'].cpu())
ref_embeddings = torch.cat(ref_embeddings_list, dim=0)
bank = NormalReferenceBank(k=min(cfg.knn_k, ref_embeddings.shape[0])).fit(ref_embeddings)

trainer.save_checkpoint(checkpoint_path, reference_bank=bank)
print('checkpoint', checkpoint_path, 'step', trainer.step, f"reference bank: {bank.embeddings.shape[0]} normal files")


In [ ]:
model.eval()
ref_batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in train_batch.items()}
with torch.no_grad():
    reference_output = model(ref_batch_device)
if bank.embeddings is None:
    bank.fit(reference_output['file_embedding'])
inference = RepresentationInference(model, bank, patchifier, masking_config=cfg)
scores = inference.score_batch(val_batch)
effective_lambda = history[-1]['effective_lambda'] if history else trainer.criterion.lambda_schedule.lambda_at(trainer.step)
print('normal train reference rows', bank.embeddings.shape[0], 'step', trainer.step, 'lambda_max', params.lambda_max, 'effective_lambda', effective_lambda)
print('validation S_pred', scores['S_pred'].tolist(), 'S_pop', scores['S_pop'].tolist(), 'timestep localization', scores['timestep_scores'][0].tolist())